# schema-tuned-sql — Remote NVIDIA Tesla T4 Runner

Executes execution-verified SFT (LoRA 1.5B FP16 and QLoRA 3B 4-bit NF4), evaluation on the frozen 220-query test set and 200-query general SQL holdout, and T4 serving benchmarks on an NVIDIA Tesla T4 (sm_75, 16 GB VRAM).

**Hardware Target**: Single NVIDIA Tesla T4 (sm_75, 16 GB VRAM). Precision: FP16 (`float16`), Attention: `sdpa`, Quantization: `4-bit NF4` (QLoRA).

The first code cell enters the repo before any shell command. On Kaggle, turn **Internet** on so it can clone into `/kaggle/working/schema-tuned-sql`, or attach the repo as Input data (that path is read-only, so the cell copies it into `/kaggle/working`). `/kaggle/working` itself is not a git checkout; `git status` / `git pull` there stop at the `/kaggle` mount.

In [ ]:
import os
import shutil
import subprocess

import torch

REPO_URL = "https://github.com/adityavsb/schema-tuned-sql.git"
REPO_NAME = "schema-tuned-sql"
KAGGLE_WORKING = "/kaggle/working"
KAGGLE_INPUT = "/kaggle/input"


def _is_repo(path: str) -> bool:
    return os.path.isfile(os.path.join(path, "pyproject.toml")) and os.path.isdir(
        os.path.join(path, "src", "train")
    )


def _walk_parents(start: str) -> str | None:
    path = os.path.abspath(start)
    while True:
        if _is_repo(path):
            return path
        parent = os.path.dirname(path)
        if parent == path or path in {KAGGLE_WORKING, "/kaggle"}:
            return None
        path = parent


def _search_kaggle_input() -> str | None:
    if not os.path.isdir(KAGGLE_INPUT):
        return None
    for entry in sorted(os.listdir(KAGGLE_INPUT)):
        base = os.path.join(KAGGLE_INPUT, entry)
        if _is_repo(base):
            return base
        if not os.path.isdir(base):
            continue
        for child in sorted(os.listdir(base)):
            nested = os.path.join(base, child)
            if _is_repo(nested):
                return nested
    return None


def _ensure_local_git(repo: str) -> None:
    """Give file copies a .git so later `git rev-parse` does not walk to /kaggle."""
    if os.path.exists(os.path.join(repo, ".git")):
        return
    print("No .git directory. Creating a local snapshot so git probes stay in this folder.")
    commands = [
        ["git", "init", "-q", "-b", "main"],
        ["git", "config", "user.email", "kaggle@local"],
        ["git", "config", "user.name", "kaggle"],
        ["git", "commit", "--allow-empty", "-q", "-m", "kaggle snapshot"],
    ]
    for cmd in commands:
        result = subprocess.run(cmd, cwd=repo)
        if result.returncode != 0:
            print(f"Local git snapshot skipped ({' '.join(cmd)} exited {result.returncode}).")
            return


found = None
search_roots = [os.getcwd()]
if os.path.isdir(KAGGLE_WORKING):
    search_roots.append(KAGGLE_WORKING)
for root in search_roots:
    for candidate in (root, os.path.join(root, REPO_NAME)):
        if _is_repo(candidate):
            found = os.path.abspath(candidate)
            break
    if found is not None:
        break
if found is None:
    found = _walk_parents(os.getcwd())
if found is None:
    found = _search_kaggle_input()

if found is not None and os.path.abspath(found).startswith(KAGGLE_INPUT + os.sep):
    if not os.path.isdir(KAGGLE_WORKING):
        raise RuntimeError(f"Found {found}, but {KAGGLE_WORKING} is not writable.")
    dest = os.path.join(KAGGLE_WORKING, REPO_NAME)
    if not _is_repo(dest):
        print(f"Copying read-only input {found} to {dest}")
        if os.path.isdir(dest):
            shutil.rmtree(dest)
        shutil.copytree(found, dest)
    found = dest

if found is None:
    parent = KAGGLE_WORKING if os.path.isdir(KAGGLE_WORKING) else os.getcwd()
    dest = os.path.join(parent, REPO_NAME)
    if os.path.isdir(dest):
        print(f"Removing incomplete directory {dest}")
        shutil.rmtree(dest)
    print(f"Cloning {REPO_URL}")
    clone_env = os.environ.copy()
    clone_env["GIT_TERMINAL_PROMPT"] = "0"
    clone = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, dest],
        env=clone_env,
    )
    if clone.returncode != 0 or not _is_repo(dest):
        raise RuntimeError(
            "git clone did not produce a checkout. Turn Internet On in the Kaggle notebook "
            "settings and re-run this cell, or attach the repository as Input data. "
            "Do not run git status or git pull in /kaggle/working: that folder is not a "
            "repository, and git stops at the /kaggle mount."
        )
    found = dest

_ensure_local_git(found)
os.chdir(found)
print(f"Working Directory: {os.getcwd()}")
assert torch.cuda.is_available(), "CUDA device not detected! Enable GPU T4 x1 in Kaggle notebook settings."

cc_major, cc_minor = torch.cuda.get_device_capability(0)
device_name = torch.cuda.get_device_name(0)
total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
print(f"GPU: {device_name} (sm_{cc_major}{cc_minor}, {total_mem_gb:.2f} GB VRAM)")

OUT_DIR = "t4_results"
os.makedirs(OUT_DIR, exist_ok=True)
!nvidia-smi

In [ ]:
# Remove outdated pre-installed torchao and install verified dependencies
!pip uninstall -y -q torchao
!pip install -q duckdb sqlglot trl peft bitsandbytes accelerate pydantic pyyaml

In [ ]:
# Build/verify customer DuckDB warehouse (11 normalized B2B SaaS tables, seed=42)
!python warehouse/generate_data.py --seed 42 --db-path warehouse/analytics.duckdb

# Check that verified 2,000-pair distillation training split is present on disk
import os
assert os.path.exists("data/train/train_2000.jsonl"), "data/train/train_2000.jsonl not found! Check repo upload."
print("Training dataset verified:")
!ls -lh data/train/train_2000.jsonl

In [ ]:
# 1. Fine-Tune S1: Qwen2.5-Coder-1.5B FP16 LoRA (r=16, alpha=32, target_modules q,k,v,o,gate,up,down)
# Expected Runtime on Tesla T4: ~10-14 minutes (2 epochs, batch size 4, grad accum 8)
!python -m src.train.sft --config configs/sft_qwen15b_lora.yaml --seed 42

In [ ]:
# 2. Fine-Tune S2: Qwen2.5-Coder-3B 4-bit QLoRA (NF4, double quant, FP16 compute, r=16, alpha=32)
# Expected Runtime on Tesla T4: ~18-24 minutes (2 epochs, batch size 4, grad accum 8)
!python -m src.train.sft --config configs/sft_qwen3b_qlora.yaml --seed 42

In [ ]:
# 3. Run CUDA Evaluation across Baselines (B0, B2) and Students (S1, S2)
# Evaluates on frozen 220-item test set (EX accuracy, 2000-resample bootstrap CIs, McNemar tests)
# and 200-item cross-domain general SQL holdout (catastrophic forgetting)
# Expected Runtime on Tesla T4: ~12-15 minutes
!python -m src.eval.runner \
    --config configs/eval.yaml \
    --mode cuda_live \
    --systems B0,B2,S1,S2 \
    --out t4_results/student_eval_summary.json

In [ ]:
# 4. Run T4 Serving Latency & Throughput Benchmark
# Measures p50/p95 latency, batch throughput (QPS, tok/s), and peak VRAM across Batch Sizes 1, 8, 32
# Computes cost per 1k queries and break-even volume against frontier API
# Expected Runtime on Tesla T4: ~2-3 minutes
!python -m src.serve.bench \
    --config configs/serve.yaml \
    --mode cuda_live \
    --out t4_results/serving_cost_analysis_t4.json

In [ ]:
# 5. Check generated outputs and package into a downloadable zip file
!ls -lh t4_results/
!zip -r t4_results.zip t4_results/ .artifacts/checkpoints/
print("\n[✓] Telemetry and checkpoints packaged into t4_results.zip. Download this file from the Kaggle sidebar.")